# ARC Phase 2 Arbitration Policy Pipeline

This notebook launches the full phase-2 pipeline: SFT with `accelerate`, GRPO with `accelerate`, validation on `data/val.jsonl`, and final testing on `data/test.jsonl`. The notebook does not create validation or test splits from the GRPO training file.

## 1. Runtime Parameters

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/beryl-07/arc.git"
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR).endswith("/notebooks"):
    PROJECT_DIR = PROJECT_DIR.parent
if Path("/kaggle/working").exists():
    PROJECT_DIR = Path("/kaggle/working/content/arc")

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
SFT_TRAIN_PATH = PROJECT_DIR / "data" / "sft_execution_data.jsonl"
GRPO_TRAIN_PATH = PROJECT_DIR / "data" / "grpo_train.jsonl"
VAL_PATH = PROJECT_DIR / "data" / "val.jsonl"
TEST_PATH = PROJECT_DIR / "data" / "test.jsonl"
SFT_OUTPUT_DIR = PROJECT_DIR / "models" / "sft_arbitration_policy"
GRPO_OUTPUT_DIR = PROJECT_DIR / "models" / "grpo_arbitration_policy"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

FORCE_RECLONE = False
RUN_SFT = True
RUN_GRPO = True
RUN_VALIDATION = True
RUN_TEST = True

NUM_PROCESSES = max(1, torch.cuda.device_count())
USE_MULTI_GPU = NUM_PROCESSES > 1
SFT_PER_DEVICE_BATCH = 4
SFT_GRAD_ACCUM = 4
GRPO_NUM_GENERATIONS = 4 if NUM_PROCESSES >= 2 else 2
GRPO_PER_DEVICE_BATCH = max(1, GRPO_NUM_GENERATIONS // NUM_PROCESSES)
GRPO_GRAD_ACCUM = 4

ACCELERATE_MIXED_PRECISION = "no"

print("Project:", PROJECT_DIR)
print("GPUs:", NUM_PROCESSES)
print("Accelerate mixed precision:", ACCELERATE_MIXED_PRECISION)

## 2. Repository and Dependencies

In [ ]:
if Path("/kaggle/working").exists():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "git", "git-lfs", "wget"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if FORCE_RECLONE and PROJECT_DIR.exists():
        import shutil
        shutil.rmtree(PROJECT_DIR)
    if PROJECT_DIR.exists():
        subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Ready:", PROJECT_DIR)

## 3. Preflight Data Check

In [ ]:
import json

for path in [SFT_TRAIN_PATH, GRPO_TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as handle:
        n = sum(1 for line in handle if line.strip())
    print(f"{path.relative_to(PROJECT_DIR)}: {n} rows")

with SFT_TRAIN_PATH.open("r", encoding="utf-8") as handle:
    first_sft = json.loads(next(handle))
if "messages" not in first_sft:
    raise ValueError("SFT training file must contain conversational `messages` records.")

## 4. SFT Training

In [ ]:
def accelerate_prefix():
    cmd = ["accelerate", "launch", "--num_processes", str(NUM_PROCESSES), "--mixed_precision", ACCELERATE_MIXED_PRECISION]
    if USE_MULTI_GPU:
        cmd.insert(2, "--multi_gpu")
    return cmd

if RUN_SFT:
    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_sft_arbitration.py"),
        "--model-name", MODEL_NAME,
        "--train-path", str(SFT_TRAIN_PATH),
        "--output-dir", str(SFT_OUTPUT_DIR),
        "--per-device-train-batch-size", str(SFT_PER_DEVICE_BATCH),
        "--gradient-accumulation-steps", str(SFT_GRAD_ACCUM),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
else:
    print("Skipping SFT.")

## 5. GRPO Training

In [ ]:
if RUN_GRPO:
    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_grpo_arbitration.py"),
        "--sft-model-path", str(SFT_OUTPUT_DIR),
        "--train-path", str(GRPO_TRAIN_PATH),
        "--output-dir", str(GRPO_OUTPUT_DIR),
        "--num-generations", str(GRPO_NUM_GENERATIONS),
        "--per-device-train-batch-size", str(GRPO_PER_DEVICE_BATCH),
        "--gradient-accumulation-steps", str(GRPO_GRAD_ACCUM),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
else:
    print("Skipping GRPO.")

## 6. Validation and Test

In [ ]:
def run_eval(split_name, data_path):
    output_path = OUTPUTS_DIR / f"arbitration_{split_name}_predictions.json"
    cmd = [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "evaluate_arbitration_policy.py"),
        "--model-path", str(GRPO_OUTPUT_DIR),
        "--data-path", str(data_path),
        "--output-path", str(output_path),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

if RUN_VALIDATION:
    run_eval("val", VAL_PATH)
if RUN_TEST:
    run_eval("test", TEST_PATH)